# Accesibilidad QroBus hacia la estación Corregidora

Este notebook identifica:

1. **Paradas de origen** desde las que se puede llegar a la estación Corregidora en 30 minutos o menos.
2. **Itinerarios de rutas** que permiten llegar dentro del umbral, tanto directos como con transbordos.
3. Un único mapa interactivo con capas consistentes para paradas directas, paradas con transbordo y trayectos representativos.

El cálculo siempre se realiza **hacia Corregidora**. Los datos base provienen del GTFS local de QroBus en `data/`.


## 1. Metodología y alcance

Se construye un grafo dirigido con cada par consecutivo de paradas de los viajes GTFS. El tiempo de cada segmento es la mediana de los tiempos programados para la combinación `origen-destino-ruta`.

Para calcular caminos con transbordos se aplica Dijkstra sobre estados `(parada, ruta)`. Cambiar de ruta agrega una penalización configurable mediante `PENALIZACION_TRANSBORDO_MIN`.

> La penalización representa espera y caminata de conexión de forma aproximada. Un cálculo exacto necesita horarios por fecha, tiempos de espera y, preferentemente, GTFS-Realtime. El notebook no presenta esta estimación como telemetría real.

Si existe `data/tiempos_corregidos_google.csv`, se suma el atraso promedio disponible al tiempo de red. Si no existe, el análisis funciona únicamente con GTFS y lo indica explícitamente.


## 2. Correcciones realizadas durante QA

Se eliminaron:

- tres mapas parcialmente duplicados;
- análisis en sentido contrario, desde Corregidora hacia otras paradas;
- dependencias no necesarias como GeoPandas, Shapely, SciPy y Matplotlib;
- rutas geométricas construidas con un viaje representativo que podía pertenecer a la dirección incorrecta;
- bloques repetidos y capturas silenciosas de excepciones.

También se corrigió el modelo de transbordos: cambiar de ruta ya no tiene costo cero.


## 3. Configuración

Los parámetros se leen desde `.env`:

```dotenv
UMBRAL_ANALISIS_MIN=30
PENALIZACION_TRANSBORDO_MIN=5
```

`RutasQroBus.ipynb` no llama a Google Maps. Solo consume la corrección ya generada por `PromedioRutas.ipynb`, cuando está disponible.


In [1]:
from pathlib import Path
from collections import defaultdict
from itertools import count
import heapq
import os
import warnings

import folium
import numpy as np
import pandas as pd
from IPython.display import display

# Funciona al ejecutar desde scripts/ o desde la raíz del repositorio.
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
if not DATA_DIR.exists():
    raise FileNotFoundError("No se encontró la carpeta local data/.")

PROJECT_ROOT = DATA_DIR.parent.resolve()
ENV_PATH = PROJECT_ROOT / ".env"


def cargar_env(env_path):
    """Carga pares CLAVE=VALOR sencillos sin imprimir secretos."""
    if not env_path.exists():
        return False

    for numero_linea, linea_original in enumerate(
        env_path.read_text(encoding="utf-8").splitlines(), start=1
    ):
        linea = linea_original.strip()
        if not linea or linea.startswith("#"):
            continue
        if linea.startswith("export "):
            linea = linea[7:].strip()
        if "=" not in linea:
            raise ValueError(f"Línea inválida en .env: {numero_linea}")

        clave, valor = linea.split("=", 1)
        clave, valor = clave.strip(), valor.strip()
        if len(valor) >= 2 and valor[0] == valor[-1] and valor[0] in {"'", '"'}:
            valor = valor[1:-1]
        os.environ.setdefault(clave, valor)

    return True


ENV_CARGADO = cargar_env(ENV_PATH)
UMBRAL_ANALISIS_MIN = float(os.getenv("UMBRAL_ANALISIS_MIN", "30"))
PENALIZACION_TRANSBORDO_MIN = float(
    os.getenv("PENALIZACION_TRANSBORDO_MIN", "5")
)

if UMBRAL_ANALISIS_MIN <= 0:
    raise ValueError("UMBRAL_ANALISIS_MIN debe ser mayor que cero.")
if PENALIZACION_TRANSBORDO_MIN < 0:
    raise ValueError("PENALIZACION_TRANSBORDO_MIN no puede ser negativa.")

CORREGIDORA_LAT = 20.600611
CORREGIDORA_LON = -100.402184

print(f"Datos: {DATA_DIR.resolve()}")
print(f".env cargado: {ENV_CARGADO}")
print(f"Umbral: {UMBRAL_ANALISIS_MIN:g} min")
print(f"Penalización por transbordo: {PENALIZACION_TRANSBORDO_MIN:g} min")


Datos: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data
.env cargado: True
Umbral: 30 min
Penalización por transbordo: 5 min


## 4. Lectura y validación del GTFS

Los identificadores se cargan como texto para conservar valores como `005`. Antes de calcular se comprueba el esquema mínimo, la unicidad de paradas y la relación entre viajes y rutas.


In [2]:
stops = pd.read_csv(DATA_DIR / "stops.txt", dtype={"stop_id": "string"})
routes = pd.read_csv(
    DATA_DIR / "routes.txt",
    dtype={"route_id": "string", "route_short_name": "string"},
)
trips = pd.read_csv(
    DATA_DIR / "trips.txt",
    dtype={"trip_id": "string", "route_id": "string"},
)
stop_times = pd.read_csv(
    DATA_DIR / "stop_times.txt",
    dtype={"trip_id": "string", "stop_id": "string"},
)

columnas_requeridas = {
    "stops": {"stop_id", "stop_name", "stop_lat", "stop_lon"},
    "routes": {"route_id", "route_short_name", "route_long_name"},
    "trips": {"trip_id", "route_id"},
    "stop_times": {
        "trip_id",
        "stop_id",
        "stop_sequence",
        "arrival_time",
        "departure_time",
    },
}
tablas = {
    "stops": stops,
    "routes": routes,
    "trips": trips,
    "stop_times": stop_times,
}

for nombre, requeridas in columnas_requeridas.items():
    faltantes = requeridas - set(tablas[nombre].columns)
    if faltantes:
        raise ValueError(f"{nombre} no contiene: {sorted(faltantes)}")

if stops["stop_id"].duplicated().any():
    raise ValueError("stops.txt contiene stop_id duplicados.")
if trips["trip_id"].duplicated().any():
    raise ValueError("trips.txt contiene trip_id duplicados.")

stop_times["stop_sequence"] = pd.to_numeric(
    stop_times["stop_sequence"], errors="raise"
)
for columna in ["stop_lat", "stop_lon"]:
    stops[columna] = pd.to_numeric(stops[columna], errors="raise")

rutas_desconocidas = set(trips["route_id"]) - set(routes["route_id"])
if rutas_desconocidas:
    raise ValueError(
        f"Hay route_id en trips.txt que no existen en routes.txt: "
        f"{sorted(rutas_desconocidas)[:10]}"
    )

print(
    f"{len(stops):,} paradas, {len(routes):,} rutas, "
    f"{len(trips):,} viajes y {len(stop_times):,} horarios."
)


2,694 paradas, 217 rutas, 32,536 viajes y 1,326,009 horarios.


## 5. Segmentos y tiempos programados

GTFS permite horas superiores a 24:00. La conversión conserva correctamente esos valores. Para cada viaje se enlaza una parada con la siguiente y se calcula:

```text
tiempo del segmento = llegada a la siguiente parada − salida de la parada actual
```

Se descartan segmentos negativos o superiores a tres horas porque indican datos incompatibles con un trayecto urbano entre paradas consecutivas.


In [3]:
def gtfs_time_to_seconds(series):
    partes = series.astype("string").str.extract(
        r"^(?P<h>\d+):(?P<m>[0-5]\d):(?P<s>[0-5]\d)$"
    )
    invalidos = partes.isna().any(axis=1)
    if invalidos.any():
        ejemplos = series[invalidos].head().tolist()
        raise ValueError(f"Horarios GTFS inválidos. Ejemplos: {ejemplos}")

    partes = partes.astype(int)
    return partes["h"] * 3600 + partes["m"] * 60 + partes["s"]


st = stop_times.copy()
st["arrival_sec"] = gtfs_time_to_seconds(st["arrival_time"])
st["departure_sec"] = gtfs_time_to_seconds(st["departure_time"])
st = st.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left",
    validate="many_to_one",
)
st = st.sort_values(["trip_id", "stop_sequence"], kind="stable")

st["next_stop_id"] = st.groupby("trip_id", sort=False)["stop_id"].shift(-1)
st["next_arrival_sec"] = st.groupby(
    "trip_id", sort=False
)["arrival_sec"].shift(-1)

segmentos_crudos = st.dropna(
    subset=["next_stop_id", "next_arrival_sec", "route_id"]
).copy()
segmentos_crudos["tiempo_segmento_min"] = (
    segmentos_crudos["next_arrival_sec"] - segmentos_crudos["departure_sec"]
) / 60

segmentos_validos = segmentos_crudos[
    segmentos_crudos["tiempo_segmento_min"].between(
        0, 180, inclusive="both"
    )
].copy()
segmentos_descartados = len(segmentos_crudos) - len(segmentos_validos)

segmentos = (
    segmentos_validos.groupby(
        ["stop_id", "next_stop_id", "route_id"], as_index=False
    )["tiempo_segmento_min"]
    .median()
)

print(f"Segmentos dirigidos únicos: {len(segmentos):,}")
print(f"Registros de segmentos descartados: {segmentos_descartados:,}")


Segmentos dirigidos únicos: 7,751
Registros de segmentos descartados: 0


## 6. Destino: estación Corregidora

Las coordenadas del proyecto se comparan con todas las paradas mediante distancia Haversine. El nodo destino del grafo es la parada QroBus más cercana. El marcador del mapa conserva las coordenadas exactas de la estación.


In [4]:
def haversine_km(lat1, lon1, lat2, lon2):
    radio_tierra_km = 6371.0088
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    a = (
        np.sin(delta_phi / 2) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2) ** 2
    )
    return radio_tierra_km * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


distancias_estacion = haversine_km(
    stops["stop_lat"].to_numpy(),
    stops["stop_lon"].to_numpy(),
    CORREGIDORA_LAT,
    CORREGIDORA_LON,
)
target_idx = int(np.argmin(distancias_estacion))
target_row = stops.iloc[target_idx]
corregidora_stop_id = target_row["stop_id"]
distancia_estacion_stop_m = float(distancias_estacion[target_idx] * 1000)

print(
    f"Parada destino: {corregidora_stop_id} — {target_row['stop_name']} "
    f"({distancia_estacion_stop_m:.0f} m de las coordenadas de la estación)."
)


Parada destino: 3025 — Estío/Calle Dr. Manuel Domínguez (211 m de las coordenadas de la estación).


## 7. Camino mínimo con transbordos

El grafo se recorre en sentido inverso desde Corregidora. El estado guarda tanto la parada como la ruta usada en el siguiente segmento. Así es posible detectar cada cambio de ruta y agregar la penalización de transbordo exactamente una vez.

El algoritmo conserva estados alternativos por ruta en una misma parada; reducir prematuramente a un solo estado produciría caminos incorrectos para sus predecesores.


In [5]:
adj_reverse = defaultdict(list)
for fila in segmentos.itertuples(index=False):
    adj_reverse[fila.next_stop_id].append(
        (fila.stop_id, float(fila.tiempo_segmento_min), fila.route_id)
    )

target_state = (corregidora_stop_id, None)
dist_state = {target_state: 0.0}
parent_state = {}
parent_step = {}
contador_heap = count()
heap = [(0.0, next(contador_heap), target_state)]

while heap:
    costo_actual, _, estado_actual = heapq.heappop(heap)
    if costo_actual > dist_state.get(estado_actual, np.inf):
        continue

    parada_actual, ruta_hacia_destino = estado_actual
    for parada_anterior, tiempo_segmento, route_id in adj_reverse.get(
        parada_actual, []
    ):
        penalizacion = (
            PENALIZACION_TRANSBORDO_MIN
            if ruta_hacia_destino is not None
            and route_id != ruta_hacia_destino
            else 0.0
        )
        nuevo_estado = (parada_anterior, route_id)
        nuevo_costo = costo_actual + tiempo_segmento + penalizacion

        if nuevo_costo < dist_state.get(nuevo_estado, np.inf):
            dist_state[nuevo_estado] = nuevo_costo
            parent_state[nuevo_estado] = estado_actual
            parent_step[nuevo_estado] = {
                "tiempo_segmento_min": tiempo_segmento,
                "penalizacion_transbordo_min": penalizacion,
            }
            heapq.heappush(
                heap, (nuevo_costo, next(contador_heap), nuevo_estado)
            )

best_state_by_stop = {corregidora_stop_id: target_state}
for estado, costo in dist_state.items():
    stop_id = estado[0]
    mejor_estado = best_state_by_stop.get(stop_id)
    if mejor_estado is None or costo < dist_state[mejor_estado]:
        best_state_by_stop[stop_id] = estado

print(f"Estados evaluados: {len(dist_state):,}")
print(f"Paradas alcanzables hacia Corregidora: {len(best_state_by_stop):,}")


Estados evaluados: 7,223
Paradas alcanzables hacia Corregidora: 2,259


In [6]:
route_short_map = routes.set_index("route_id")[
    "route_short_name"
].to_dict()
stop_name_map = stops.set_index("stop_id")["stop_name"].to_dict()


def compactar_rutas(rutas_segmentos):
    compactas = []
    for route_id in rutas_segmentos:
        if not compactas or route_id != compactas[-1]:
            compactas.append(route_id)
    return compactas


def reconstruir_camino(stop_id):
    estado = best_state_by_stop[stop_id]
    paradas_camino = [stop_id]
    rutas_segmentos = []
    guard = {estado}

    while estado != target_state:
        if estado not in parent_state:
            raise RuntimeError(f"Camino incompleto para la parada {stop_id}.")

        rutas_segmentos.append(estado[1])
        estado = parent_state[estado]
        if estado in guard:
            raise RuntimeError(f"Ciclo inesperado para la parada {stop_id}.")
        guard.add(estado)
        paradas_camino.append(estado[0])

    rutas_compactas = compactar_rutas(rutas_segmentos)
    paradas_transbordo = [
        paradas_camino[indice]
        for indice in range(1, len(rutas_segmentos))
        if rutas_segmentos[indice] != rutas_segmentos[indice - 1]
    ]
    return paradas_camino, rutas_segmentos, rutas_compactas, paradas_transbordo


registros = []
for stop_id, mejor_estado in best_state_by_stop.items():
    (
        paradas_camino,
        rutas_segmentos,
        rutas_compactas,
        paradas_transbordo,
    ) = reconstruir_camino(stop_id)

    etiquetas_rutas = [
        route_short_map.get(route_id, route_id) for route_id in rutas_compactas
    ]
    registros.append(
        {
            "stop_id": stop_id,
            "tiempo_red_min": dist_state[mejor_estado],
            "num_transbordos": max(len(rutas_compactas) - 1, 0),
            "num_paradas_camino": len(paradas_camino),
            "tipo_conexion": (
                "Destino"
                if stop_id == corregidora_stop_id
                else "Directa"
                if len(rutas_compactas) <= 1
                else "Con transbordo"
            ),
            "itinerario_route_ids": " → ".join(rutas_compactas),
            "itinerario_rutas": " → ".join(etiquetas_rutas),
            "paradas_transbordo": " → ".join(
                stop_name_map.get(s, s) for s in paradas_transbordo
            ),
            "camino_stop_ids": paradas_camino,
            "rutas_por_segmento": rutas_segmentos,
        }
    )

resultados = (
    pd.DataFrame(registros)
    .merge(
        stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]],
        on="stop_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("tiempo_red_min")
    .reset_index(drop=True)
)

display(
    resultados[
        [
            "stop_id",
            "stop_name",
            "tiempo_red_min",
            "num_transbordos",
            "itinerario_rutas",
        ]
    ].head(10)
)


,stop_id,stop_name,tiempo_red_min,num_transbordos,itinerario_rutas
0,3025,Estío/Calle Dr. Manuel Domínguez,0.000000,0,
1,2341,Av. Felipe Ángeles/Av. San Roque,1.533333,0,C54
2,3295,Felipe Ángeles/Fraternidad,2.216667,0,C54
3,2340,Av. Felipe Ángeles/Plan de Ayala Poniente,2.733333,0,C54
4,3294,Av. Felipe Ángeles/Calle del Porvenir,3.583333,0,C54
5,2339,Av. Felipe Ángeles/Felipe Ángeles 225,4.200000,0,C54
6,2338,Av. Felipe Ángeles/Estadística,4.700000,0,C54
7,4256,Epigmenio González/Departamental Parques,5.366667,0,C54
8,3363,Prol. Tecnológico/Calle San Joaquín,8.033333,0,C54
9,3362,Prol. Tecnológico/Ford Citelis Querétaro,9.200000,0,C54


## 8. Corrección opcional con observaciones de Google

La corrección se aplica únicamente cuando la fila está marcada como disponible. Se suma `atraso_promedio_ruta_min` al tiempo de red calculado aquí; no se reemplaza el camino ni se modifica el número de transbordos.

Las filas sin observación continúan como `GTFS programado`. Esta separación evita presentar datos no observados como si provinieran de Google.


In [7]:
correcciones_path = DATA_DIR / "tiempos_corregidos_google.csv"

if correcciones_path.exists():
    correcciones = pd.read_csv(
        correcciones_path, dtype={"stop_id": "string"}
    )
    requeridas = {
        "stop_id",
        "atraso_promedio_ruta_min",
        "correccion_disponible",
    }
    faltantes = requeridas - set(correcciones.columns)
    if faltantes:
        raise ValueError(
            f"El archivo de correcciones no contiene: {sorted(faltantes)}"
        )

    correcciones = correcciones[
        [
            "stop_id",
            "atraso_promedio_ruta_min",
            "correccion_disponible",
        ]
    ].drop_duplicates("stop_id", keep="last")

    disponible = correcciones["correccion_disponible"]
    correcciones["correccion_disponible"] = (
        disponible.eq(True)
        | disponible.astype(str).str.lower().eq("true")
    )
    correcciones["atraso_promedio_ruta_min"] = pd.to_numeric(
        correcciones["atraso_promedio_ruta_min"], errors="coerce"
    )

    resultados = resultados.merge(
        correcciones,
        on="stop_id",
        how="left",
        validate="one_to_one",
    )
else:
    resultados["atraso_promedio_ruta_min"] = np.nan
    resultados["correccion_disponible"] = False

resultados["correccion_disponible"] = resultados[
    "correccion_disponible"
].fillna(False).astype(bool)
resultados["atraso_aplicado_min"] = (
    resultados["atraso_promedio_ruta_min"]
    .where(resultados["correccion_disponible"], 0)
    .fillna(0)
    .clip(lower=0)
)
resultados["tiempo_estimado_min"] = (
    resultados["tiempo_red_min"] + resultados["atraso_aplicado_min"]
)
resultados["fuente_tiempo"] = np.where(
    resultados["correccion_disponible"],
    "GTFS + corrección Google",
    "GTFS programado",
)

cobertura_google = resultados.loc[
    resultados["stop_id"].ne(corregidora_stop_id),
    "correccion_disponible",
].mean()

print(
    f"Corrección Google encontrada: {correcciones_path.exists()}. "
    f"Cobertura de paradas alcanzables: {cobertura_google:.1%}."
)


Corrección Google encontrada: True. Cobertura de paradas alcanzables: 88.9%.


## 9. Resultados dentro de 30 minutos

La parada destino no se cuenta como origen. Una parada se clasifica dentro del umbral usando `tiempo_estimado_min`, que incluye:

- tiempo mediano de segmentos GTFS;
- penalización por cada transbordo;
- atraso de Google, solo cuando está disponible.


In [8]:
paradas_30 = resultados[
    resultados["stop_id"].ne(corregidora_stop_id)
    & resultados["tiempo_estimado_min"].le(UMBRAL_ANALISIS_MIN)
].copy()

paradas_directas_30 = paradas_30[
    paradas_30["num_transbordos"].eq(0)
].copy()
paradas_con_transbordo_30 = paradas_30[
    paradas_30["num_transbordos"].gt(0)
].copy()

tabla_itinerarios_30 = (
    paradas_30.groupby(
        [
            "itinerario_route_ids",
            "itinerario_rutas",
            "tipo_conexion",
            "num_transbordos",
        ],
        as_index=False,
    )
    .agg(
        paradas_origen=("stop_id", "nunique"),
        tiempo_minimo_min=("tiempo_estimado_min", "min"),
        tiempo_maximo_min=("tiempo_estimado_min", "max"),
        tiempo_promedio_min=("tiempo_estimado_min", "mean"),
    )
    .sort_values(
        ["num_transbordos", "tiempo_minimo_min", "itinerario_rutas"]
    )
    .reset_index(drop=True)
)

print(f"Paradas de origen dentro de {UMBRAL_ANALISIS_MIN:g} min: {len(paradas_30):,}")
print(f"  Conexión directa: {len(paradas_directas_30):,}")
print(f"  Con transbordos: {len(paradas_con_transbordo_30):,}")
print(f"Itinerarios de rutas distintos: {len(tabla_itinerarios_30):,}")


Paradas de origen dentro de 30 min: 33
  Conexión directa: 15
  Con transbordos: 18
Itinerarios de rutas distintos: 9


### 9.1 Rutas e itinerarios que llegan dentro del umbral

Cada fila representa una secuencia de rutas, no una geometría duplicada por cada parada. `C54` indica un viaje directo; `CXX → C54` indica un transbordo.


In [9]:
columnas_itinerarios = [
    "itinerario_rutas",
    "tipo_conexion",
    "num_transbordos",
    "paradas_origen",
    "tiempo_minimo_min",
    "tiempo_promedio_min",
    "tiempo_maximo_min",
]
display(
    tabla_itinerarios_30[columnas_itinerarios].style.format(
        {
            "tiempo_minimo_min": "{:.1f}",
            "tiempo_promedio_min": "{:.1f}",
            "tiempo_maximo_min": "{:.1f}",
        }
    )
)


,itinerario_rutas,tipo_conexion,num_transbordos,paradas_origen,tiempo_minimo_min,tiempo_promedio_min,tiempo_maximo_min
0,C54,Directa,0,15,7.0,15.6,29.2
1,T07 → C54,Con transbordo,1,4,22.6,25.3,28.4
2,L154 → C54,Con transbordo,1,3,24.3,27.6,29.9
3,C62 → C54,Con transbordo,1,2,25.8,26.9,28.1
4,C33 → C54,Con transbordo,1,5,27.3,28.3,29.1
5,T03 → C54,Con transbordo,1,1,29.8,29.8,29.8
6,C24 → C33 → C54,Con transbordo,2,1,27.9,27.9,27.9
7,C34 → C62 → C54,Con transbordo,2,1,28.6,28.6,28.6
8,C63 → T07 → C54,Con transbordo,2,1,29.1,29.1,29.1


### 9.2 Paradas dentro del umbral, incluyendo transbordos

La tabla conserva el itinerario, los puntos de transbordo y la fuente del tiempo. Esto permite distinguir estimaciones GTFS de aquellas que ya incorporan una corrección de Google.


In [10]:
tabla_paradas_30 = paradas_30[
    [
        "stop_id",
        "stop_name",
        "tipo_conexion",
        "itinerario_rutas",
        "paradas_transbordo",
        "num_transbordos",
        "num_paradas_camino",
        "tiempo_red_min",
        "atraso_aplicado_min",
        "tiempo_estimado_min",
        "fuente_tiempo",
    ]
].sort_values(["tiempo_estimado_min", "stop_name"])

display(
    tabla_paradas_30.style.format(
        {
            "tiempo_red_min": "{:.1f}",
            "atraso_aplicado_min": "{:.1f}",
            "tiempo_estimado_min": "{:.1f}",
        }
    )
)


,stop_id,stop_name,tipo_conexion,itinerario_rutas,paradas_transbordo,num_transbordos,num_paradas_camino,tiempo_red_min,atraso_aplicado_min,tiempo_estimado_min,fuente_tiempo
1,2341,Av. Felipe Ángeles/Av. San Roque,Directa,C54,,0,2,1.5,5.4,7.0,GTFS + corrección Google
2,3295,Felipe Ángeles/Fraternidad,Directa,C54,,0,3,2.2,5.4,7.6,GTFS + corrección Google
3,2340,Av. Felipe Ángeles/Plan de Ayala Poniente,Directa,C54,,0,4,2.7,5.4,8.2,GTFS + corrección Google
4,3294,Av. Felipe Ángeles/Calle del Porvenir,Directa,C54,,0,5,3.6,5.4,9.0,GTFS + corrección Google
5,2339,Av. Felipe Ángeles/Felipe Ángeles 225,Directa,C54,,0,6,4.2,5.4,9.6,GTFS + corrección Google
6,2338,Av. Felipe Ángeles/Estadística,Directa,C54,,0,7,4.7,5.4,10.1,GTFS + corrección Google
7,4256,Epigmenio González/Departamental Parques,Directa,C54,,0,8,5.4,5.4,10.8,GTFS + corrección Google
8,3363,Prol. Tecnológico/Calle San Joaquín,Directa,C54,,0,9,8.0,5.4,13.5,GTFS + corrección Google
9,3362,Prol. Tecnológico/Ford Citelis Querétaro,Directa,C54,,0,10,9.2,5.4,14.6,GTFS + corrección Google
10,2889,Av. Cerro Sombrerete/Av. Playa Roqueta,Directa,C54,,0,11,11.3,5.4,16.7,GTFS + corrección Google


## 10. Pruebas automáticas de calidad

Estas comprobaciones validan que los caminos terminen en Corregidora, que los tiempos sean no negativos, que los transbordos coincidan con los cambios de ruta y que ninguna parada del reporte exceda el umbral.


In [11]:
assert segmentos["tiempo_segmento_min"].ge(0).all()
assert corregidora_stop_id in best_state_by_stop
assert resultados["stop_id"].is_unique
assert np.isfinite(resultados["tiempo_red_min"]).all()
assert resultados["tiempo_red_min"].ge(0).all()
assert resultados["atraso_aplicado_min"].ge(0).all()
assert paradas_30["tiempo_estimado_min"].le(
    UMBRAL_ANALISIS_MIN + 1e-9
).all()
assert paradas_30["stop_id"].ne(corregidora_stop_id).all()

for fila in resultados.itertuples(index=False):
    assert fila.camino_stop_ids[-1] == corregidora_stop_id
    rutas_compactas = compactar_rutas(fila.rutas_por_segmento)
    assert fila.num_transbordos == max(len(rutas_compactas) - 1, 0)
    assert len(fila.camino_stop_ids) == len(fila.rutas_por_segmento) + 1

print("QA OK: todas las validaciones automáticas fueron superadas.")


QA OK: todas las validaciones automáticas fueron superadas.


## 11. Mapa único de accesibilidad

El mapa consolida las visualizaciones anteriores:

- **Azul:** paradas con conexión directa.
- **Naranja:** paradas que requieren uno o más transbordos.
- **Rojo:** estación Corregidora y su parada GTFS más cercana.
- **Trayectos representativos:** un camino por cada itinerario de rutas, para evitar cientos de líneas duplicadas.

El control de capas permite ocultar o mostrar cada grupo sin crear mapas inconsistentes.


In [12]:
mapa_30 = folium.Map(
    location=[CORREGIDORA_LAT, CORREGIDORA_LON],
    zoom_start=13,
    tiles="OpenStreetMap",
)

capa_directas = folium.FeatureGroup(
    name=f"Paradas directas ({len(paradas_directas_30)})",
    show=True,
)
capa_transbordos = folium.FeatureGroup(
    name=f"Paradas con transbordo ({len(paradas_con_transbordo_30)})",
    show=True,
)
capa_trayectos = folium.FeatureGroup(
    name=f"Trayectos representativos ({len(tabla_itinerarios_30)})",
    show=True,
)

folium.Marker(
    [CORREGIDORA_LAT, CORREGIDORA_LON],
    tooltip="Estación Corregidora",
    popup=(
        f"<b>Estación Corregidora</b><br>"
        f"Parada GTFS cercana: {target_row['stop_name']}<br>"
        f"Distancia: {distancia_estacion_stop_m:.0f} m"
    ),
    icon=folium.Icon(color="red", icon="flag"),
).add_to(mapa_30)

folium.CircleMarker(
    [target_row["stop_lat"], target_row["stop_lon"]],
    radius=5,
    color="darkred",
    fill=True,
    fill_opacity=0.9,
    tooltip=f"Parada destino GTFS: {target_row['stop_name']}",
).add_to(mapa_30)

for fila in paradas_directas_30.itertuples(index=False):
    folium.CircleMarker(
        [fila.stop_lat, fila.stop_lon],
        radius=4,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.75,
        tooltip=(
            f"{fila.stop_name} | {fila.tiempo_estimado_min:.1f} min | "
            f"{fila.itinerario_rutas}"
        ),
    ).add_to(capa_directas)

for fila in paradas_con_transbordo_30.itertuples(index=False):
    folium.CircleMarker(
        [fila.stop_lat, fila.stop_lon],
        radius=4,
        color="orange",
        fill=True,
        fill_color="orange",
        fill_opacity=0.8,
        tooltip=(
            f"{fila.stop_name} | {fila.tiempo_estimado_min:.1f} min | "
            f"{fila.itinerario_rutas} | "
            f"{fila.num_transbordos} transbordo(s)"
        ),
    ).add_to(capa_transbordos)

representantes = (
    paradas_30.sort_values("tiempo_estimado_min")
    .drop_duplicates("itinerario_route_ids", keep="first")
)

coords_map = stops.set_index("stop_id")[
    ["stop_lat", "stop_lon"]
].to_dict("index")

for fila in representantes.itertuples(index=False):
    coordenadas_camino = [
        (
            coords_map[stop_id]["stop_lat"],
            coords_map[stop_id]["stop_lon"],
        )
        for stop_id in fila.camino_stop_ids
    ]
    color = "blue" if fila.num_transbordos == 0 else "orange"
    folium.PolyLine(
        coordenadas_camino,
        color=color,
        weight=3,
        opacity=0.65,
        tooltip=(
            f"{fila.itinerario_rutas} | "
            f"{fila.tiempo_estimado_min:.1f} min"
        ),
    ).add_to(capa_trayectos)

capa_directas.add_to(mapa_30)
capa_transbordos.add_to(mapa_30)
capa_trayectos.add_to(mapa_30)
folium.LayerControl(collapsed=False).add_to(mapa_30)

coordenadas_visibles = [
    [CORREGIDORA_LAT, CORREGIDORA_LON],
    *paradas_30[["stop_lat", "stop_lon"]].to_numpy().tolist(),
]
if len(coordenadas_visibles) > 1:
    mapa_30.fit_bounds(coordenadas_visibles)

MAPA_PATH = DATA_DIR / "mapa_rutas_qrobus.html"
mapa_30.save(MAPA_PATH)
print(f"Mapa interactivo exportado: {MAPA_PATH.resolve()}")

# Al quedar como la última expresión, Jupyter integra el mapa en la salida de la celda.
# No depende del archivo HTML exportado arriba.
mapa_30


Mapa interactivo exportado: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/mapa_rutas_qrobus.html


## 12. Interpretación correcta

El resultado responde: **“¿Qué paradas tienen un camino programado hacia Corregidora cuyo tiempo mediano, penalizaciones de transbordo y corrección disponible no superan 30 minutos?”**

No responde todavía:

- cuál unidad específica llegará;
- si un transbordo concreto estará sincronizado;
- el atraso GPS real del autobús;
- qué servicio opera en una fecha particular, porque el conjunto local no incluye `calendar.txt` ni `calendar_dates.txt`.

Para operación en tiempo real debe incorporarse GTFS-Realtime `TripUpdates` y, de ser posible, `VehiclePositions`.
